In [ ]:
import copy
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.config import AE_RIDGE_GPU_CONFIG
from src.data import load_and_prep_data_strided
from src.backtest.gpu_engine_ae import run_ae_multigpu_backtest
from src.evaluation.metrics import calculate_global_metrics

np.random.seed(42)

CONFIG = copy.deepcopy(AE_RIDGE_GPU_CONFIG)
CONFIG['data_path'] = 'all30min/'
CONFIG['output_path'] = 'results_ae_ridge/ae_ridge_backtest.csv'
CONFIG['weights_dir'] = 'results_ae_ridge/ae_weights/'

# Override model/train hyperparameters as needed
# CONFIG['model']['n_components'] = 5
# CONFIG['model']['alpha_recon'] = 0.5
# CONFIG['model']['alpha_ridge'] = 1.0
# CONFIG['train']['num_epochs'] = 50
# CONFIG['train']['learning_rate'] = 1e-3
# CONFIG['train']['batch_size'] = 10
# CONFIG['gpu_count'] = 2

LOADING_HPARAMS = {
    'exog_cols': 'none',
    'use_transform_exog': False,
    'use_diurnal': True,
    'allow_missing': False,
    'use_winsor': True,
    'feature_type': 'raw',
    'lag_scope': 'global',
}

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
X_np, y_np, dates, baselines, features = load_and_prep_data_strided(
    LOADING_HPARAMS, CONFIG['data_path']
)

# Set n_features from data
CONFIG['model']['n_features'] = X_np.shape[1]

print(f'Data: {X_np.shape[0]:,} samples, {X_np.shape[1]} features')
print(f'Training window: {CONFIG["train_window"]:,} periods')
print(f'Test samples: {X_np.shape[0] - CONFIG["train_window"]:,}')

In [ ]:
os.makedirs(os.path.dirname(CONFIG['output_path']), exist_ok=True)

results = run_ae_multigpu_backtest(
    X_np, y_np, dates, baselines, CONFIG
)

In [ ]:
metrics = calculate_global_metrics(results)
print(f"QLIKE: {metrics.get('qlike', float('nan')):.4f}")
print(f"MSE:   {metrics.get('mse', float('nan')):.6f}")
print(f"MAE:   {metrics.get('mae', float('nan')):.6f}")
print(f"N:     {metrics.get('n_samples', 0):,}")

# Saved weights summary
if CONFIG.get('weights_dir') and os.path.isdir(CONFIG['weights_dir']):
    n_weights = len([f for f in os.listdir(CONFIG['weights_dir']) if f.endswith('.pt')])
    print(f'Saved {n_weights} weight snapshots to {CONFIG["weights_dir"]}')

# --- Plot 1: Time Series (last 500 points) ---
plt.figure(figsize=(12, 6))
subset = results.iloc[-500:]
plt.plot(subset.index, subset['true_raw'], label='True RV', alpha=0.6, color='black')
plt.plot(subset.index, subset['pred_raw'], label='Predicted RV', alpha=0.8, color='#1f77b4')
plt.title('Volatility Forecast: AE+Ridge vs Ground Truth')
plt.ylabel('Realized Volatility')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# --- Plot 2: Diagnostic Scatter ---
plt.figure(figsize=(10, 10))
plt.scatter(results['true_raw'], results['pred_raw'], alpha=0.3, s=10, label='Predictions')
plt.plot([1e-6, 1e-2], [1e-6, 1e-2], 'k-', alpha=0.5, label='Perfect Match')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('True Volatility')
plt.ylabel('Predicted Volatility')
plt.title(f"AE+Ridge Diagnostic: QLIKE = {metrics.get('qlike', float('nan')):.4f}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()